# Week 7 Hands-On Lab — Learning to Act, and Learning to Cheat

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime**. Trains in seconds.

- **Part A** — tabular **Q-learning** on Gymnasium FrozenLake.
- **Part B** — a **reward-hacking** diagnosis: a mis-specified reward makes the optimal policy farm a proxy instead of solving the task.
- **Part C** — dissect the hacked policy.
- **Part D** — **design and test your own reward fix.**

> **Report only what your run actually produced.** Learning curves are stochastic; quote your run's numbers.

**Attribution.** The Q-learning workflow mirrors the HuggingFace Deep RL Course (Unit 2) and Farama Gymnasium. The reward-hacking framing follows DeepMind's specification-gaming catalogue and OpenAI's *Faulty Reward Functions in the Wild*. All lab code is original to ESP3201.

## Setup

In [ ]:
import os
try:
    import numpy
except ModuleNotFoundError:
    os.system("pip install -q numpy")
try:
    import gymnasium
except ModuleNotFoundError:
    os.system("pip install -q gymnasium")

import numpy as np
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

# --- Week 7 lab core, embedded directly (no repo clone) -----------------------
# Cloning a support module from Colab is fragile: if a session already ran once
# before an update landed, the clone silently no-ops onto the stale cached copy
# instead of fetching the fix (git clone refuses to overwrite an existing,
# non-empty directory, and os.system() does not surface that failure). Pasting
# the needed code directly here removes that whole failure class -- everything
# this notebook needs lives in this one cell, nothing to go stale.
# Canonical source, kept in sync by hand: starter/rl_lab.py in the course repo.

class RewardHackGrid:
    """A 1-D corridor:  [S] [ ] [P] [ ] [G]

    - The agent starts at S (cell 0).
    - P (the "polish"/proxy tile) pays `proxy_reward` every time the agent
      *enters* it. The agent can leave and re-enter to farm it.
    - G (the true goal) ends the episode and pays `goal_reward`.
    - Each step costs `step_cost`.

    reward_mode:
      "proxy"  -> entering P pays proxy_reward (the mis-specified reward).
      "fixed"  -> entering P pays nothing; only reaching G is rewarded.

    Gymnasium-style API so the same q_learning() works here and on FrozenLake.
    Observation is the integer cell index. Actions: 0 = left, 1 = right.
    """

    def __init__(self, length: int = 5, proxy_cell: int = 2, proxy_reward: float = 1.0,
                 goal_reward: float = 5.0, step_cost: float = 0.0,
                 reward_mode: str = "proxy", max_steps: int = 50,
                 custom_reward=None):
        self.length = length
        self.proxy_cell = proxy_cell
        self.goal_cell = length - 1
        self.proxy_reward = proxy_reward
        self.goal_reward = goal_reward
        self.step_cost = step_cost
        self.reward_mode = reward_mode
        self.max_steps = max_steps
        # custom_reward(prev_pos, new_pos, entered, env) -> float, used when
        # reward_mode == "custom". Lets students design and TEST their own fix.
        self.custom_reward = custom_reward
        self.n_states = length
        self.n_actions = 2
        self.pos = 0
        self.steps = 0

    def reset(self, seed: Optional[int] = None) -> Tuple[int, Dict]:
        self.pos = 0
        self.steps = 0
        return self.pos, {}

    def step(self, action: int) -> Tuple[int, float, bool, bool, Dict]:
        self.steps += 1
        prev = self.pos
        if action == 1:
            self.pos = min(self.length - 1, self.pos + 1)
        else:
            self.pos = max(0, self.pos - 1)

        entered = self.pos != prev

        if self.reward_mode == "custom" and self.custom_reward is not None:
            # Student-authored reward fully defines the step reward.
            reward = float(self.custom_reward(prev, self.pos, entered, self))
        else:
            reward = self.step_cost
            if self.pos == self.proxy_cell and entered and self.reward_mode == "proxy":
                reward += self.proxy_reward  # <-- the reward hack lives here

        terminated = False
        if self.pos == self.goal_cell:
            if self.reward_mode != "custom":
                reward += self.goal_reward
            terminated = True

        truncated = self.steps >= self.max_steps
        info = {"reached_goal": self.pos == self.goal_cell, "cell": self.pos}
        return self.pos, reward, terminated, truncated, info


@dataclass
class StepTrace:
    """One Q-learning update, kept only for episodes a caller asked to trace."""
    step: int
    state: int
    action: int
    next_state: int
    reward: float
    q_before: float
    td_target: float
    td_error: float
    q_after: float


@dataclass
class EpisodeLog:
    ret: float
    success: bool
    steps: int
    trace: Optional[List[StepTrace]] = None


@dataclass
class TrainResult:
    Q: np.ndarray
    logs: List[EpisodeLog] = field(default_factory=list)
    # (episode_index, Q.copy()) pairs, only populated if snapshot_every was set.
    snapshots: List[Tuple[int, np.ndarray]] = field(default_factory=list)

    def returns(self) -> List[float]:
        return [e.ret for e in self.logs]

    def successes(self) -> List[int]:
        return [int(e.success) for e in self.logs]

    def moving_success_rate(self, window: int = 50) -> float:
        tail = self.successes()[-window:]
        return sum(tail) / max(1, len(tail))

    def episodes_to_reach(self, threshold: float, window: int = 50) -> Optional[int]:
        """First episode index whose trailing `window`-episode success rate
        reaches `threshold`, or None if it never does. Used to compare how
        fast different hyperparameters converge, not just where they end up."""
        succ = self.successes()
        for i in range(len(succ)):
            tail = succ[max(0, i - window + 1): i + 1]
            if sum(tail) / len(tail) >= threshold:
                return i
        return None


def q_learning(env, n_states: int, n_actions: int, episodes: int = 2000,
               alpha: float = 0.1, gamma: float = 0.95,
               epsilon: float = 1.0, epsilon_min: float = 0.05,
               epsilon_decay: float = 0.999, max_steps: int = 100,
               is_success: Optional[Callable] = None,
               seed: int = 0,
               snapshot_every: Optional[int] = None,
               trace_episodes: Optional[set] = None) -> TrainResult:
    """Standard epsilon-greedy tabular Q-learning.

    is_success(terminated, reward, info) -> bool defines TASK success, which is
    intentionally separate from reward. Defaults to "terminated with positive
    reward" (works for FrozenLake) but RewardHackGrid passes a goal check.

    snapshot_every: if set, record a (episode_index, Q.copy()) snapshot every
    N episodes (plus a final one), so a caller can plot how specific Q-values
    move over the course of training.

    trace_episodes: if set, record every individual TD update (state, action,
    reward, Q-value before/after, TD error) for just those episode indices --
    the "watch one trajectory update the Q-table" view. Left None for normal
    runs so training stays cheap.
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_states, n_actions), dtype=float)
    if is_success is None:
        def is_success(terminated, reward, info):
            return bool(terminated and reward > 0)

    logs: List[EpisodeLog] = []
    snapshots: List[Tuple[int, np.ndarray]] = []
    eps = epsilon
    for ep in range(episodes):
        obs, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
        total = 0.0
        success = False
        step_trace = [] if (trace_episodes and ep in trace_episodes) else None
        for t in range(max_steps):
            if rng.random() < eps:
                action = int(rng.integers(0, n_actions))
            else:
                action = int(np.argmax(Q[obs]))
            nxt, reward, terminated, truncated, info = env.step(action)
            best_next = 0.0 if terminated else np.max(Q[nxt])
            q_before = Q[obs, action]
            td_target = reward + gamma * best_next
            td_error = td_target - q_before
            Q[obs, action] += alpha * td_error
            if step_trace is not None:
                step_trace.append(StepTrace(t, obs, action, nxt, reward, q_before,
                                             td_target, td_error, Q[obs, action]))
            obs = nxt
            total += reward
            if is_success(terminated, reward, info):
                success = True
            if terminated or truncated:
                break
        logs.append(EpisodeLog(total, success, t + 1, trace=step_trace))
        if snapshot_every and ep % snapshot_every == 0:
            snapshots.append((ep, Q.copy()))
        eps = max(epsilon_min, eps * epsilon_decay)
    if snapshot_every:
        snapshots.append((episodes - 1, Q.copy()))
    return TrainResult(Q, logs, snapshots)


def greedy_policy(Q: np.ndarray) -> np.ndarray:
    return np.argmax(Q, axis=1)


def evaluate(env, Q: np.ndarray, max_steps: int = 100,
             is_success: Optional[Callable] = None) -> Dict:
    """Run the greedy policy once; report return, success, and trajectory."""
    if is_success is None:
        def is_success(terminated, reward, info):
            return bool(terminated and reward > 0)
    obs, _ = env.reset()
    total = 0.0
    success = False
    traj = [obs]
    for _ in range(max_steps):
        action = int(np.argmax(Q[obs]))
        obs, reward, terminated, truncated, info = env.step(action)
        traj.append(obs)
        total += reward
        if is_success(terminated, reward, info):
            success = True
        if terminated or truncated:
            break
    return {"return": total, "success": success, "trajectory": traj}


def make_frozenlake(slippery: bool = False):
    """Lazy import so the reward-hacking part has no Gymnasium dependency."""
    import gymnasium as gym
    return gym.make("FrozenLake-v1", is_slippery=slippery)


def reached_goal_success(terminated, reward, info):
    return bool(info.get("reached_goal", False))


print("Week 7 lab core loaded (embedded in this notebook -- no repo clone needed).")

## Part A — Q-learning on FrozenLake

### A.0 — What does this environment actually look like?

Before training anything, look at the environment itself: a 4x4 grid. `S` is the start, `G` is the
goal, `H` is a hole (falling in ends the episode with reward 0), `F` is safe ice. The agent picks
one of 4 actions each step (left/down/right/up) and moves accordingly.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_frozenlake(env, trajectory=None, title="FrozenLake"):
    """Draw the 4x4 grid (and, if given, one episode's path through it)."""
    desc = [[c.decode() for c in row] for row in env.unwrapped.desc]
    nrow, ncol = len(desc), len(desc[0])
    colors = {"S": "#a6d8ff", "F": "#eaf6ff", "H": "#2c2c2c", "G": "#ffd700"}
    fig, ax = plt.subplots(figsize=(ncol * 1.1 + 1.5, nrow * 1.1))
    for r in range(nrow):
        for c in range(ncol):
            tile = desc[r][c]
            ax.add_patch(patches.Rectangle((c, nrow - 1 - r), 1, 1, facecolor=colors[tile], edgecolor="gray"))
            ax.text(c + 0.5, nrow - 1 - r + 0.5, tile, ha="center", va="center", fontsize=14, fontweight="bold")
    if trajectory:
        xs = [s % ncol + 0.5 for s in trajectory]
        ys = [nrow - 1 - s // ncol + 0.5 for s in trajectory]
        ax.plot(xs, ys, "o-", color="crimson", linewidth=2, markersize=8, zorder=5)
        ax.plot(xs[0], ys[0], "s", color="blue", markersize=16, zorder=6, label="episode start")
        ax.plot(xs[-1], ys[-1], "*", color="black", markersize=20, zorder=6, label="episode end")
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.set_xlim(0, ncol); ax.set_ylim(0, nrow)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)
    ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

plot_frozenlake(make_frozenlake(slippery=False), title="FrozenLake -- S=start, F=frozen (safe), H=hole, G=goal")

**Now watch one episode before any learning has happened.** With no Q-table yet, the agent just picks uniformly random actions -- this is what "exploration" looks like on the ground.

In [ ]:
import numpy as np

_demo_env = make_frozenlake(slippery=False)
_rng = np.random.default_rng(0)
obs, _ = _demo_env.reset(seed=0)
random_trajectory = [obs]
for _ in range(30):
    action = int(_rng.integers(0, _demo_env.action_space.n))
    obs, reward, terminated, truncated, info = _demo_env.step(action)
    random_trajectory.append(obs)
    if terminated or truncated:
        break

outcome = "reached the goal" if reward > 0 else ("fell in a hole" if terminated else "ran out of steps")
plot_frozenlake(_demo_env, trajectory=random_trajectory,
               title=f"One random-policy episode -- {outcome}")

In [ ]:
env = make_frozenlake(slippery=False)
ns, na = env.observation_space.n, env.action_space.n
res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0)
print(f"final 100-episode success rate: {res.moving_success_rate(100):.2f}")
ev = evaluate(make_frozenlake(slippery=False), res.Q)
print(f"greedy policy: return={ev['return']:.1f}  success={ev['success']}  steps={len(ev['trajectory'])-1}")

**Compare against the trained policy's path** on the same grid -- this is the same `ev` you just computed above, just drawn on the map instead of printed as a list of state indices.

In [ ]:
plot_frozenlake(make_frozenlake(slippery=False), trajectory=ev["trajectory"],
               title=f"Trained greedy policy -- {'reached the goal' if ev['success'] else 'did not reach the goal'}")

In [ ]:
import matplotlib.pyplot as plt
succ = res.successes(); window = 100
smooth = [sum(succ[max(0,i-window):i+1]) / len(succ[max(0,i-window):i+1]) for i in range(len(succ))]
plt.figure(figsize=(7,3)); plt.plot(smooth)
plt.xlabel('episode'); plt.ylabel(f'success rate ({window}-ep moving)')
plt.title('FrozenLake Q-learning'); plt.tight_layout(); plt.show()

**Try it:** sweep `alpha`, `gamma`, `epsilon_decay`. Which most changes how fast (and whether) the agent learns?

## Part B0 — Q-learning fundamentals: watching a Q-table learn

Part A trained a Q-table and showed you a success-rate curve at the end. This section opens that
up: how do individual Q-values actually move as the agent gathers experience, and how do the
learning-rate and exploration-schedule choices change *how fast* (and whether) that happens?

Every update follows the same rule:

```
Q(s, a)  <-  Q(s, a) + alpha * ( reward + gamma * max_a' Q(s', a') - Q(s, a) )
                                \_____________ TD target _____________/
                       \_________________ TD error __________________/
```

`alpha` (learning rate) controls how much of the TD error gets absorbed into `Q(s, a)` on
each single update. `gamma` (discount) controls how much of the *next* state's best value counts
toward this one. Everything below just watches this one line run, many times, at different settings.

### B0.1 — Watch the value function propagate from the goal

Re-train on FrozenLake, but this time snapshot the whole Q-table every few hundred episodes.
Plotting `V(s) = max_a Q(s, a)` as a 4x4 heatmap at each snapshot lets you watch the *goal's* value
propagate backward through the grid as training progresses -- this is the mechanism behind that
success-rate curve in Part A, not a separate phenomenon.

In [ ]:
env = make_frozenlake(slippery=False)
ns, na = env.observation_space.n, env.action_space.n
snap_res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                      epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0,
                      snapshot_every=300)

n_snaps = len(snap_res.snapshots)
fig, axes = plt.subplots(2, (n_snaps + 1) // 2, figsize=(2.2 * ((n_snaps + 1) // 2), 4.4))
for ax, (ep, Q) in zip(axes.flat, snap_res.snapshots):
    V = Q.max(axis=1).reshape(4, 4)
    im = ax.imshow(V, vmin=0, vmax=1, cmap="viridis")
    succ_so_far = snap_res.successes()[:ep + 1]
    window = succ_so_far[-100:]
    ax.set_title(f"ep {ep}\nsucc={sum(window)/len(window):.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[n_snaps:]:
    ax.axis("off")
fig.suptitle("V(s) = max_a Q(s,a) across training -- watch the goal's value spread backward")
fig.colorbar(im, ax=axes, shrink=0.6, label="V(s)")
plt.show()

**Look for:** which cells light up first, and whether that matches the cells right next to the goal. Cells that never light up are usually holes or cells the greedy policy never needs to pass through.

### B0.2 — Watch individual trajectories update the Q-table

Two single episodes, traced step by step: one from partway through training (still noisy,
still updating a lot), one from near the end (mostly converged, small TD errors). Same update
rule, very different behavior.

In [ ]:
env = make_frozenlake(slippery=False)
traced_res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                        epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0,
                        trace_episodes={650, 2600})

def show_trace(ep_idx, label):
    ep_log = traced_res.logs[ep_idx]
    print(f"--- episode {ep_idx} ({label}) -- success={ep_log.success}, {len(ep_log.trace)} steps ---")
    for st in ep_log.trace[:10]:
        print(f"  step {st.step:2d}: state {st.state:2d} -> action {st.action} -> reward {st.reward:+.2f} | "
              f"Q before {st.q_before:6.3f}  TD target {st.td_target:6.3f}  "
              f"TD error {st.td_error:+.3f}  ->  Q after {st.q_after:6.3f}")
    print()

show_trace(650, "mid-training")
show_trace(2600, "near-converged")

In [ ]:
def trace_trajectory(ep_log):
    """Reconstruct the sequence of visited cells from a traced episode's step log --
    same idea as plot_frozenlake's `trajectory` argument, just built from the trace
    instead of from evaluate()."""
    states = [st.state for st in ep_log.trace]
    states.append(ep_log.trace[-1].next_state)
    return states

for ep_idx, label in [(650, "mid-training"), (2600, "near-converged")]:
    ep_log = traced_res.logs[ep_idx]
    outcome = "reached goal" if ep_log.success else "did not reach goal"
    plot_frozenlake(env, trajectory=trace_trajectory(ep_log),
                   title=f"episode {ep_idx} ({label}) -- {outcome}, {len(ep_log.trace)} steps")

### Checkpoint D -- what does a converged update look like?

Compare the TD errors in the two traces. What happens to the TD error as a state-action pair gets visited many times under a stable policy? Does a near-zero TD error mean the Q-value is *correct*, or only that it stopped changing?

### B0.3 — Zoomed out: how do alpha and the exploration schedule affect convergence?

Instead of one run, train several runs that each vary *one* setting at a time and plot the learning
curve (a rolling success rate over episodes) for each. Overlaying these curves is a more direct read
than a single summary number: you can see not just *whether* a setting converges but *how* -- fast
and steady, slow and steady, or noisy and unstable.

In [ ]:
def rolling_success_rate(successes, window=100):
    """Trailing moving average, one value per episode -- the y-axis for the plot below."""
    succ = np.asarray(successes, dtype=float)
    out = np.empty(len(succ))
    for i in range(len(succ)):
        lo = max(0, i - window + 1)
        out[i] = succ[lo:i + 1].mean()
    return out

FIXED_ALPHA = 0.1        # used while sweeping epsilon_decay
FIXED_DECAY = 0.999      # used while sweeping alpha
SWEEP_EPISODES = 3000
alphas = [0.02, 0.05, 0.1, 0.3, 0.6]
epsilon_decays = [0.995, 0.998, 0.999, 0.9995, 0.9998]

alpha_curves = {}
for a in alphas:
    sweep_env = make_frozenlake(slippery=False)
    sweep_res = q_learning(sweep_env, ns, na, episodes=SWEEP_EPISODES, alpha=a, gamma=0.95,
                           epsilon=1.0, epsilon_decay=FIXED_DECAY, max_steps=100, seed=0)
    alpha_curves[a] = rolling_success_rate(sweep_res.successes(), window=100)

decay_curves = {}
for ed in epsilon_decays:
    sweep_env = make_frozenlake(slippery=False)
    sweep_res = q_learning(sweep_env, ns, na, episodes=SWEEP_EPISODES, alpha=FIXED_ALPHA, gamma=0.95,
                           epsilon=1.0, epsilon_decay=ed, max_steps=100, seed=0)
    decay_curves[ed] = rolling_success_rate(sweep_res.successes(), window=100)

print(f"trained {len(alphas)} alpha runs (epsilon_decay={FIXED_DECAY} fixed) "
      f"and {len(epsilon_decays)} epsilon_decay runs (alpha={FIXED_ALPHA} fixed), "
      f"{SWEEP_EPISODES} episodes each.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

cmap_a = plt.cm.viridis(np.linspace(0.15, 0.9, len(alphas)))
for color, a in zip(cmap_a, alphas):
    axes[0].plot(alpha_curves[a], color=color, label=f"alpha={a}")
axes[0].axhline(0.8, color="gray", linestyle=":", linewidth=1)
axes[0].set_title(f"Varying alpha (epsilon_decay={FIXED_DECAY} fixed)")
axes[0].set_xlabel("episode"); axes[0].set_ylabel("success rate (rolling, window=100)")
axes[0].legend(fontsize=8, loc="lower right")

cmap_d = plt.cm.plasma(np.linspace(0.15, 0.9, len(epsilon_decays)))
for color, ed in zip(cmap_d, epsilon_decays):
    axes[1].plot(decay_curves[ed], color=color, label=f"epsilon_decay={ed}")
axes[1].axhline(0.8, color="gray", linestyle=":", linewidth=1)
axes[1].set_title(f"Varying epsilon_decay (alpha={FIXED_ALPHA} fixed)")
axes[1].set_xlabel("episode")
axes[1].legend(fontsize=8, loc="lower right")

fig.suptitle("How alpha and the exploration schedule shape the learning curve (dotted line = 80% threshold)")
plt.tight_layout(); plt.show()

### Checkpoint E -- reading the curves

- Which `epsilon_decay` curve never crosses the 80% line, and why -- too much residual randomness late in training, or too little exploration early on?
- Is there a decay value whose curve is clearly *worse* than both a faster- and a slower-decaying one? What does that tell you about "more exploration is always safer"?
- Do the alpha curves separate from each other as much as the epsilon_decay curves do? Would you expect that ranking to hold in every environment, or is it specific to FrozenLake's sparse, deterministic reward?

## Part B — A reward you can hack

`RewardHackGrid` is a corridor `[S][ ][P][ ][G]`. Under the **proxy** reward, stepping into the polish tile `P` pays out *every time*. Watch **reward earned** climb while **task success** stays at zero.

In [ ]:
def plot_reward_grid(env, trajectory=None, title="RewardHackGrid"):
    """Draw the 1-D corridor and, if given, one episode's path through it --
    same idea as plot_frozenlake(), adapted to a 1-row corridor instead of a 4x4 grid."""
    labels = ["F"] * env.length
    labels[0] = "S"
    labels[env.proxy_cell] = "P"
    labels[env.goal_cell] = "G"
    colors = {"S": "#a6d8ff", "F": "#eaf6ff", "P": "#ffb86b", "G": "#ffd700"}
    fig, ax = plt.subplots(figsize=(env.length * 1.3 + 1.5, 2.3))
    for c in range(env.length):
        ax.add_patch(patches.Rectangle((c, 0), 1, 1, facecolor=colors[labels[c]], edgecolor="gray"))
        ax.text(c + 0.5, 0.85, labels[c], ha="center", va="center", fontsize=14, fontweight="bold")
    if trajectory:
        xs = [s + 0.5 for s in trajectory]
        # vertical position climbs steadily over the episode, so repeated back-and-forth
        # visits to the same cell (farming the proxy tile) show up as a rising zigzag
        # instead of stacking exactly on top of each other.
        n = len(trajectory)
        ys = [0.12 + 0.5 * (i / max(1, n - 1)) for i in range(n)]
        ax.plot(xs, ys, "o-", color="crimson", linewidth=1.2, markersize=6, alpha=0.7, zorder=5)
        ax.plot(xs[0], ys[0], "s", color="blue", markersize=14, zorder=6, label="episode start")
        ax.plot(xs[-1], ys[-1], "*", color="black", markersize=18, zorder=6, label="episode end")
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.set_xlim(0, env.length); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)
    ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

plot_reward_grid(RewardHackGrid(reward_mode="proxy"),
                 title="RewardHackGrid -- S=start, F=plain corridor, P=exploitable 'polish' tile, G=goal")

In [ ]:
proxy_env = RewardHackGrid(reward_mode='proxy')
hacked = q_learning(proxy_env, proxy_env.n_states, proxy_env.n_actions,
                    episodes=1500, max_steps=proxy_env.max_steps,
                    is_success=reached_goal_success, seed=0)
final_return = sum(hacked.returns()[-100:]) / 100.0
final_success = hacked.moving_success_rate(100)
print(f"MIS-SPECIFIED reward:  mean return={final_return:.1f}   goal-success rate={final_success:.2f}")
ev = evaluate(RewardHackGrid(reward_mode='proxy'), hacked.Q, is_success=reached_goal_success)
print(f"greedy trajectory (cells): {ev['trajectory']}")
print('^ it oscillates around the proxy cell and never reaches cell 4 (G).')

In [ ]:
plot_reward_grid(RewardHackGrid(reward_mode="proxy"), trajectory=ev["trajectory"],
                 title="MIS-SPECIFIED reward, greedy policy -- oscillates on P, never reaches G")

### The built-in fix

`reward_mode='fixed'` removes the proxy payout entirely -- **that alone is what stops the hack.**
Nothing rewards revisiting `P` anymore, so a policy that farms it earns nothing extra for doing so.

`step_cost=-0.05` is a separate, secondary choice: a small per-step penalty. It is easy to assume
the fix is "remove the proxy payout *and* add a step cost" as one combined change, but the step
cost is not what stops the hacking. Don't take that on faith -- the next two cells check it.

In [ ]:
# Does the step cost matter for reaching the goal directly, or is removing
# the proxy payout doing all the work on its own?
for step_cost in (0.0, -0.05):
    trial_env = RewardHackGrid(reward_mode='fixed', step_cost=step_cost)
    trial_res = q_learning(trial_env, trial_env.n_states, trial_env.n_actions,
                           episodes=1500, max_steps=trial_env.max_steps,
                           is_success=reached_goal_success, seed=0)
    trial_ev = evaluate(RewardHackGrid(reward_mode='fixed', step_cost=step_cost), trial_res.Q,
                        is_success=reached_goal_success)
    print(f"step_cost={step_cost:+.2f}:  success={trial_ev['success']}  "
          f"trajectory={trial_ev['trajectory']}")

**Both step costs reach the goal by the direct path.** The step cost changes the *return* you'd
report, not whether the hack is fixed -- with or without it, removing the proxy payout is enough.

Now check the reverse: does adding a step cost, *without* removing the proxy payout, stop the hack?

In [ ]:
# Keep the mis-specified (proxy) reward, but add the same step cost the "fix" uses.
# If the step cost were doing the real work, this should also stop the hacking.
still_proxy_env = RewardHackGrid(reward_mode='proxy', step_cost=-0.05)
still_proxy_res = q_learning(still_proxy_env, still_proxy_env.n_states, still_proxy_env.n_actions,
                             episodes=1500, max_steps=still_proxy_env.max_steps,
                             is_success=reached_goal_success, seed=0)
still_proxy_ev = evaluate(RewardHackGrid(reward_mode='proxy', step_cost=-0.05), still_proxy_res.Q,
                          is_success=reached_goal_success)
print(f"proxy reward + step_cost=-0.05:  success={still_proxy_ev['success']}  "
      f"trajectory={still_proxy_ev['trajectory'][:15]}...")
print("Still hacked -- the step cost alone does not fix a mis-specified reward.")

### Checkpoint -- what actually fixed it?

State, in one sentence, exactly which change stopped the hacking (not "we changed the reward
function" -- name the specific term that was removed). What is the step cost actually for, if not
fixing the hack?

In [ ]:
fixed_env = RewardHackGrid(reward_mode='fixed', step_cost=-0.05)
fixed = q_learning(fixed_env, fixed_env.n_states, fixed_env.n_actions,
                   episodes=1500, max_steps=fixed_env.max_steps,
                   is_success=reached_goal_success, seed=0)
ev2 = evaluate(RewardHackGrid(reward_mode='fixed', step_cost=-0.05), fixed.Q, is_success=reached_goal_success)
print(f"FIXED reward:  greedy return={ev2['return']:.2f}  success={ev2['success']}  trajectory={ev2['trajectory']}")

**Same corridor, contrast the path.** The fixed policy below should go straight from `S` to `G` with no back-and-forth on `P`.

In [ ]:
plot_reward_grid(RewardHackGrid(reward_mode="fixed", step_cost=-0.05), trajectory=ev2["trajectory"],
                 title="FIXED reward, greedy policy -- direct path to G")

## Part C — Dissect the hacked policy

In [ ]:
print('Greedy action per cell (0=left, 1=right):', greedy_policy(hacked.Q))
for s in range(proxy_env.n_states):
    print(f'  cell {s}: left={hacked.Q[s,0]:7.2f}  right={hacked.Q[s,1]:7.2f}')
print('\nReward spec: entering cell', proxy_env.proxy_cell, 'pays', proxy_env.proxy_reward,
      'every time; goal (cell', proxy_env.goal_cell, ') pays', proxy_env.goal_reward, 'once.')

## Part D — Design and TEST your own reward fix

Don't just propose a fix — implement it and check whether the trained policy still hacks. Write a `custom_reward(prev_pos, new_pos, entered, env)` that returns the reward for one step, then train with `reward_mode='custom'`.

A fix *works* only if the greedy policy **reaches the goal** (`success=True`). Try to make one that does — and try a tempting one that secretly still rewards the proxy and watch it fail.

### Worked example: capping instead of deleting

Part B's fix (`reward_mode='fixed'`) worked by **deleting** the proxy payout entirely. That's only
possible because the proxy tile was pure duplicate information -- the agent's position is already
directly measurable, so nothing was lost by dropping the term. Most real proxies aren't like that:
a code-review approval, a click-through rate, a grader's score are often the *only* cheap signal
you have. You can't just delete them without losing your only feedback.

A more practical fix: keep the proxy reward, but make it **idempotent** -- pay it out at most once
per episode, so re-entering the tile stops being profitable. This is a toy version of a real
technique (de-duplicating or capping a reward signal so a repeated cheap trigger can't be farmed --
e.g. only counting the *first* approval on a given diff, not every resubmission after cosmetic
changes).

In [ ]:
class ProxyOnceState:
    """Tracks whether the proxy tile has already paid out THIS episode."""
    def __init__(self):
        self.paid_this_episode = False

def make_proxy_once_reward():
    """Cap the proxy payout to once per episode instead of deleting it. Unlike
    reward_mode='fixed', this keeps the proxy signal -- it just stops it from
    being repeatedly farmable within a single episode. `env.steps == 1` marks
    the first step of a fresh episode (RewardHackGrid resets it in reset()),
    which is when the cap resets."""
    state = ProxyOnceState()
    def proxy_once_reward(prev_pos, new_pos, entered, env):
        if env.steps == 1:
            state.paid_this_episode = False
        r = -0.05
        if new_pos == env.proxy_cell and entered and not state.paid_this_episode:
            r += 1.0
            state.paid_this_episode = True
        if new_pos == env.goal_cell:
            r += 5.0
        return r
    return proxy_once_reward, state

capped_reward, _ = make_proxy_once_reward()
capped_env = RewardHackGrid(reward_mode='custom', custom_reward=capped_reward)
capped_res = q_learning(capped_env, capped_env.n_states, capped_env.n_actions, episodes=1500,
                        max_steps=capped_env.max_steps, is_success=reached_goal_success, seed=0)

# A fresh reward closure for evaluation -- eval runs its own episode(s), so it
# needs its own ProxyOnceState rather than reusing the one already consumed by
# training's 1500 episodes.
eval_reward, _ = make_proxy_once_reward()
capped_ev = evaluate(RewardHackGrid(reward_mode='custom', custom_reward=eval_reward), capped_res.Q,
                     is_success=reached_goal_success)
print(f"CAPPED PROXY (idempotent, not deleted):  success={capped_ev['success']}  "
      f"return={capped_ev['return']:.2f}  trajectory={capped_ev['trajectory']}")

In [ ]:
plot_reward_grid(RewardHackGrid(reward_mode="proxy"), trajectory=capped_ev["trajectory"],
                 title="Capped-proxy fix, greedy policy -- direct path, proxy still paid (once)")

### Checkpoint -- capping vs. deleting

Compare this trajectory to Part B's FIXED-reward trajectory. Both reach the goal directly, but this
one still pays the proxy reward once per episode instead of removing the term. Why does capping the
payout stop the hack just as effectively as deleting it? What real-world proxy could you cap this
way (once per attempt, once per user, once per time window) instead of deleting -- precisely because
deleting it isn't an option?

Now try designing your own fix below -- capping is one option, not the only one.

In [ ]:
def my_reward(prev_pos, new_pos, entered, env):
    # EDIT THIS. Available: env.proxy_cell, env.goal_cell, env.length.
    # Example starting point (a small step cost + a goal bonus, no proxy payout):
    r = -0.05
    if new_pos == env.goal_cell:
        r += 5.0
    return r

env = RewardHackGrid(reward_mode='custom', custom_reward=my_reward)
res = q_learning(env, env.n_states, env.n_actions, episodes=1500,
                 max_steps=env.max_steps, is_success=reached_goal_success, seed=0)
ev = evaluate(RewardHackGrid(reward_mode='custom', custom_reward=my_reward), res.Q,
              is_success=reached_goal_success)
print(f"YOUR reward:  reached_goal={ev['success']}  return={ev['return']:.2f}  trajectory={ev['trajectory']}")
print('A working fix makes reached_goal=True.')

## Worksheet (your deliverable)

### 1. Reward-vs-success table

| Reward design | Mean return | Goal-success rate |
|---------------|-------------|-------------------|
| proxy (mis-specified) | | |
| fixed (built-in) | | |
| **your custom fix** | | |

### 2. Diagnose the hack

- Which cell/action does the hacked greedy policy exploit, and what in the **reward spec** makes it profitable? Cite the Q-values.
- Why is *mean return* a misleading success metric here?

### 3. Your fix

- Paste your `custom_reward`. Did it make `reached_goal=True`?
- If your first attempt failed, what did the policy do instead, and what did you change?
- `What you re-measured to confirm the fix (not just higher reward):`

### 4. Connect to deployment

- In one sentence: where might a real robot reward function hide a proxy like this one?

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking